In [ ]:
#@title 1 · Install VoxCPM + API + tunnel
!pip install -q voxcpm fastapi "uvicorn[standard]" python-multipart soundfile requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("deps installed")


In [ ]:
#@title 2 · Write the API server
%%writefile /content/server.py
import os
import soundfile as sf
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from voxcpm import VoxCPM

# emotion preset -> VoxCPM natural-language style prefix (the model reads it as control, not speech)
EMOTION_STYLE = {
    "default": "",
    "friendly": "(warm, friendly tone)",
    "cheerful": "(cheerful, upbeat tone)",
    "excited": "(excited, energetic, fast tone)",
    "sad": "(sad, slow, melancholic tone)",
    "angry": "(angry, intense tone)",
    "terrified": "(terrified, fearful, trembling)",
    "shouting": "(shouting loudly)",
    "whispering": "(whispering softly)",
}
EMOTIONS = list(EMOTION_STYLE)

print("[init] loading VoxCPM2 (first run downloads the model, ~3-6 min) ...")
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SR = model.tts_model.sample_rate
print("[init] ready. sample_rate =", SR)

app = FastAPI(title="VoxCPM One-Shot Clone API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "device": "cuda", "model": "VoxCPM2", "emotions": EMOTIONS}

@app.post("/clone")
async def clone(text: str = Form(...), emotion: str = Form("default"),
                speed: float = Form(1.0), reference: UploadFile = File(...)):
    if emotion not in EMOTION_STYLE:
        raise HTTPException(400, f"emotion must be one of {EMOTIONS}")
    if not text.strip():
        raise HTTPException(400, "text is empty")
    suffix = os.path.splitext(reference.filename or "ref.wav")[1] or ".wav"
    ref = f"/content/ref_input{suffix}"
    with open(ref, "wb") as f:
        f.write(await reference.read())
    pace = "" if 0.9 <= speed <= 1.1 else ("(slower) " if speed < 0.9 else "(faster) ")
    style = EMOTION_STYLE[emotion]
    prompt = (style + pace + text) if (style or pace) else text
    wav = model.generate(text=prompt, reference_wav_path=ref,
                         cfg_value=2.0, inference_timesteps=10)
    out = "/content/cloned.wav"
    sf.write(out, wav, SR)
    return FileResponse(out, media_type="audio/wav", filename="cloned.wav")


In [ ]:
#@title 3 · Launch API + print PUBLIC_API_URL
import os, re, threading, subprocess, uvicorn
os.chdir("/content")
def _run(): uvicorn.run("server:app", host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_run, daemon=True).start()
# start the public tunnel immediately; the model keeps loading in the thread
proc = subprocess.Popen(["cloudflared","tunnel","--url","http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
URL=None
for line in proc.stdout:
    m=re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m: URL=m.group(0); break
print("\n"+"="*62)
print("  PUBLIC_API_URL=" + str(URL))
print("="*62)
print(">> URL is live now, but VoxCPM2 is still loading (~3-6 min on first run);")
print(">> the backend returns errors until you see '[init] ready' above.")
